In [1]:
import sys
from pathlib import Path

# Encontra o módulo e disponibiliza seu pacote Python, sem depender da pasta atual.
PASTA_ATUAL = Path.cwd().resolve()
PASTA_PROJETO = next(
    (
        pasta
        for base in (PASTA_ATUAL, *PASTA_ATUAL.parents)
        for pasta in (
            base,
            base / "04_regressao_polinomial",
            base / "regressao_modelos_lineares" / "04_regressao_polinomial",
        )
        if (pasta / "src" / "polinomial" / "config.py").is_file()
    ),
    None,
)
if PASTA_PROJETO is None:
    raise FileNotFoundError("Não foi possível localizar o módulo 04_regressao_polinomial.")
PASTA_SRC = PASTA_PROJETO / "src"
if str(PASTA_SRC) not in sys.path:
    sys.path.insert(0, str(PASTA_SRC))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.preprocessing import (
    OrdinalEncoder,
    OneHotEncoder,
    PolynomialFeatures,
    PowerTransformer,
    StandardScaler,
    QuantileTransformer,
)

from polinomial.auxiliares import dataframe_coeficientes
from polinomial.config import DADOS_DIABETES_CATEGORIZADOS
from polinomial.graficos import plot_comparar_metricas_modelos, plot_coeficientes
from polinomial.modelos import RANDOM_STATE
from polinomial.modelos import grid_search_cv_regressor, organiza_resultados, treinar_e_validar_modelo_regressao

sns.set_theme(palette="bright")

In [ ]:
df = pd.read_parquet(DADOS_DIABETES_CATEGORIZADOS)

df.head()

In [ ]:
df.info()

In [ ]:
X = df.drop(columns="target")
y = df["target"]

In [ ]:
colunas_power_transform = ["imc", "ldl", "hdl", "colesterol_total"]

coluna_target = ["target"]

coluna_ordinal_encoder = ["colesterol_hdl_cat"]

coluna_one_hot_encoder = ["sexo"]

colunas_standard_scaler = [
    coluna
    for coluna in df.columns
    if coluna not in colunas_power_transform + coluna_target + coluna_ordinal_encoder + coluna_one_hot_encoder
]

colunas_standard_scaler

In [ ]:
categorias_ordinal_encoder = [
    ["2-3", "4-5", "6+"],
]

In [ ]:
preprocessamento_categoricas = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ],
    remainder="passthrough",
)

preprocessamento_simples = ColumnTransformer(
    transformers=[
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
        ("standard_scaler", StandardScaler(), X.columns.difference(coluna_ordinal_encoder + coluna_one_hot_encoder)),
    ],
    remainder="passthrough",
)

preprocessamento_completo = ColumnTransformer(
    [
        ("power_transform", PowerTransformer(method="box-cox"), colunas_power_transform),
        ("standard_scaler", StandardScaler(), colunas_standard_scaler),
        ("ordinal_encoder", OrdinalEncoder(categories=categorias_ordinal_encoder), coluna_ordinal_encoder),
        ("one_hot_encoder", OneHotEncoder(drop="if_binary"), coluna_one_hot_encoder),
    ]
)

target_transformer = QuantileTransformer(n_quantiles=20, output_distribution="normal")

In [ ]:
elasticnet_regressor = ElasticNet()

In [ ]:
param_grid = {
    "regressor__reg__alpha": [1E-2, 5E-2, 0.1, 1.0, 5.0, 10],
    "regressor__reg__l1_ratio": [0.1, 0.25, 0.5, 0.75, 0.9],
    "regressor__preprocessor": [preprocessamento_categoricas, preprocessamento_simples, preprocessamento_completo],
    "transformer": [target_transformer, None]
}

In [ ]:
grid_search = grid_search_cv_regressor(
    regressor=elasticnet_regressor,
    param_grid=param_grid,
    preprocessor=preprocessamento_categoricas,
    target_transformer=target_transformer,
)

grid_search

In [ ]:
grid_search.fit(X, y)

In [ ]:
grid_search.best_params_

In [ ]:
grid_search.best_score_

In [ ]:
grid_search.best_estimator_

In [ ]:
print(*grid_search.cv_results_.keys(), sep="\n")

In [ ]:
max(grid_search.cv_results_["mean_test_neg_root_mean_squared_error"])

In [ ]:
grid_search.best_estimator_

In [ ]:
grid_search.best_estimator_.regressor_["preprocessor"]

In [ ]:
grid_search.best_estimator_.regressor_["reg"]

In [ ]:
grid_search.best_estimator_.transformer_

In [ ]:
grid_search.best_estimator_.regressor_["reg"].coef_

In [ ]:
grid_search.best_estimator_.regressor_["reg"].intercept_

In [ ]:
grid_search.best_estimator_.regressor_["preprocessor"].get_feature_names_out()

In [ ]:
coefs = dataframe_coeficientes(
    grid_search.best_estimator_.regressor_["reg"].coef_,
    grid_search.best_estimator_.regressor_["preprocessor"].get_feature_names_out()
)

coefs

In [ ]:
plot_coeficientes(coefs, "ElasticNet com StdSc, OE, OHE")

In [ ]:
regressors = {
    "DummyRegressor": {
        "preprocessor": None,
        "regressor": DummyRegressor(strategy="mean"),
        "target_transformer": None,
    },
    
    "LinearRegression_pre_simples": {
        "preprocessor": preprocessamento_simples,
        "regressor": LinearRegression(),
        "target_transformer": None,
    },
    "ElasticNet_grid_search": {
        "preprocessor": grid_search.best_estimator_.regressor_["preprocessor"],
        "regressor": grid_search.best_estimator_.regressor_["reg"],
        "target_transformer": grid_search.best_estimator_.transformer_,
    },
}

In [ ]:
resultados = {
    nome_modelo: treinar_e_validar_modelo_regressao(X, y, **regressor)
    for nome_modelo, regressor in regressors.items()
}

resultados

In [ ]:
df_resultados = organiza_resultados(resultados)

df_resultados

In [ ]:
plot_comparar_metricas_modelos(df_resultados)